# Example: Run a grid of `MESA` models using `subprocesses`

In this example, we will see how to use the `morpy` package to prepare and run a grid of `MESA` models with `Python`. Specifically, we will see how to run the grid using the built-in `concurrent.futures` module. This will allow you to run your models in parallel on single-node machines. To run a grid using `MPI` (via `mpi4pi`), which allows you to run a grid over multiple nodes (i.e. on a cluster), please look at the `run_a_grid_with_mpi4pi` example.

Before reading this example, it is recommended to first go through the example called `prepare_and_run_single_model`.

As in the aforementioned example, we will look at the blocking (run) and non-blocking (launch) ways to compute `MESA` models in parallel. Since these grid computations can take a long time, the latter option is the more attractive one now. Therefore, we will also see how to track the status of our models, even when the `Python` session that launched them was terminated.

## 1. Imports

In [ ]:
from pathlib import Path # We use Path objects instead of strings for cleaner-looking paths to fiels and directories 
import morpy as morpy # The morpy package. We import the whole package for convenience, but separate impprts for the modules is also possible.
import numpy as np

## 2. General settings

The first few lines of the code block below are used to specify the `MESA` installation that you would like to use. These are similar to the environment variables you set in your `.*rc` files during the `MESA` and `MESASDK` installation (as specified here: https://docs.mesastar.org/en/26.4.1/installation.html) or in your module files. Even though the code has the capabilities to parse your `MESA` version from the `MESA` installation directory (here `MESA_DIR`), it does not hurt to specify it explicitly. Just make sure that the specified `MESA_VERSION` matches that of the `MESA` version installed at `MESA_DIR` to avoid conflicts.


The variable `PROJECT_DIR` is where your `MESA` project will be created. A `MESA` project consists of all the necessary files to compile the `Fortran` executables (`clean`, `mk`, `make/`, and `src/`), as well as the files to specify the hostory and profile output. Each `MESA` model gets its own directory within the project direcotory, where its inlists are stored and its output is written.


In [2]:
# Adjust these paths to match your local desired MESA installation.
MESA_DIR = Path('/Users/janhenneco/software/mesa/mesa-r24.08.1')
MESASDK_ROOT = Path('/Applications/mesasdk-25.12.1/')
OMP_THREADS  = 1 # Number of threads used per MESA model. Can be specified on a model-per-model basis when running/launching the models.
MESA_VERSION = 'r24.08.1'

## 3. Set up `MESA` environment and parse default option files

In this step, which you only have to do once per project, we do two things. 

First, we set up the `MESA` environment, just as you would do in your `.*rc` or module file (e.g. `export MESA_DIR=...`), and create a `MesaConfig` object (called `mconf` in the code below). The latter holds information about the `MESA` installation, such as the `MESA` version in a normalised numeric format allowing the version-aware parts of the code to work properly. The `setup_mesa_env` function is a wrapper function calling two functions that can alsoe be used separately (e.g. when you only want to create `MESA` inlists and do no need to compile it and run models, you only need to `MesaConfig` object): MesaConfig.setup() for the MesaConfig object and MesaConfig.set_mesa_env() for a dictionary containing the runtime environment for `MESA`.

Second, we parse the `MESA` default options files. These files usually live at `MESA_DIR/<star|binary|kap|eos>/defaults`. The resulting `DefaultsOptions` file (called `def_opts` here) contains `DefaultsFile` objects for all the different available default files in the `MESA` installation directory. These `DefaultsFile` objects are can be explored, which comes in handy for creating inlists (see later).

Although many of the methods and functions used in the `morpy` package can generate the `MesaConfig` and `DefaultsOptions` objects on the fly, it is recommended to generate them once and pass them as an argument when a function or method requires it to avoid to much overhead.

In [ ]:
mconf = morpy.setup_mesa_env(mesa_dir=MESA_DIR, mesasdk_root=MESASDK_ROOT, omp_num_threads=OMP_THREADS, version_name=MESA_VERSION)
def_opts = morpy.build_defaults_options_from_config(mconf)
print('MESA version:', mconf.mesa_version.version_name)
print('     - year:', mconf.mesa_version.year)
print('     - month:', mconf.mesa_version.month)
print('     - patch:', mconf.mesa_version.patch)
print('     - scheme:', mconf.mesa_version.scheme)
print('     - revision:', mconf.mesa_version.revision)

MESA version: r24.08.1
     - year: 24
     - month: 8
     - patch: 1
     - scheme: release
     - revision: None


## 4. Prepare a model grid

In this example, we will run a small 2D grid of `MESA` models. Starting from a base model, that we import from an existing model directory, we will vary the initial mass and overshooting value.

We will put the each of the resulting `Model` objects into a `GridRunItem` object. In the next steps, these `GridRunItems` will be collected and passed onto the grid functions. Putting the `Model` objects into these `GridRunItem` objects might seem like a reduentant layer of complexity, though the reason for the existence of the `GridRunItem` object class is simple. It has two attributes only: a `Model` and a `Mapping` attribute. The latter is the same kind of object as created by the `setup_mesa_env()` function used above and holds the environment variables required for running `MESA`. By including this as an extra attribute that is passed along to the grid functions, we can specify custom envirnment variables for each model separately. We can, for example if we know that some models in our grid are heavier to run than others, assign a higher number of threads (`OMP_NUM_THREADS`) to specific models. 

In [ ]:
# Import the basic work directory from the MESA installation directory
base_model = morpy.Model.empty(name='base')
# This function automatically imports all files inside the given directory containing the word "inlist" in their filename
morpy.import_inlists_from_dir(model=base_model, source_dir=MESA_DIR / 'star' / 'work', default_options=def_opts, config=mconf)

vars(base_model)

{'name': 'base',
 'model_path': None,
 'mesa_version': MesaVersion(version_name='r24.08.1', scheme='release', revision=None, year=24, month=8, patch=1),
 'model_id': '5f1bb50353c2427b813bc3e865448054',
 'state': <ModelState.DRAFT: 'draft'>,
 'inlists': {'inlist': Inlist(name='inlist', mesa_version=MesaVersion(version_name='r24.08.1', scheme='release', revision=None, year=24, month=8, patch=1), options={'read_extra_star_job_inlist(1)': MesaOption(name='read_extra_star_job_inlist(1)', default_option='.false.', description='You can split your ``star_job`` inlist into pieces using the following controls.\nBTW: it works recursively, so the extras can read extras too.\nif ``read_extra_star_job_inlist(i)`` is true,\nthen read &star_job from this namelist file', namelist='star_job', current_value='.true.'), 'extra_star_job_inlist_name(1)': MesaOption(name='extra_star_job_inlist_name(1)', default_option="'undefined'", description='', namelist='star_job', current_value="'inlist_project'"), 'read

From the output of `vars(base_model)`, we see that the model currently has three inlists; `inlist`, `inlist_project`, and `inlist_pgstar`. Since we'll be running a grid, we are not interested in `PGPLOT` output, so we can remove this inlist from the model.

Afterwards, we'll make some changes to the remaining inlists before creating copies of this model.


In [5]:
base_model.remove_inlist(inlist_name='inlist_pgstar')

inlist_header = base_model.inlists['inlist']
inlist_project = base_model.inlists['inlist_project']

inlist_header.set_option(name='read_extra_pgstar_inlist(1)', value=False, namelist='pgstar', defaults=def_opts)
inlist_project.set_option(name='pgstar_flag', value=False, namelist='star_job', defaults=def_opts)
inlist_project.remove_option(name='save_model_filename')
inlist_project.remove_option(name='stop_near_zams')
inlist_project.remove_option(name='Lnuc_div_L_zams_limit')
inlist_project.set_options(defaults=def_opts, controls={'xa_central_lower_limit(1)': '1d-6',
                                                        'overshoot_f0(1)': 0.004,
                                                        'overshoot_bdy_loc(1)': 'top',
                                                        'overshoot_scheme(1)': 'exponential',
                                                        'overshoot_zone_loc(1)': 'core',
                                                        'overshoot_zone_type(1)': 'burn_H'})



In [ ]:
# We can use this code block to preview the changes made to the inlists above
print('Contents of `inlist`:')
to_print = morpy.preview(base_model.inlists['inlist'].options, prefer_pandas=False)
print(to_print, '\n\n')
print('Contents of `inlist_project`:')
to_print = morpy.preview(base_model.inlists['inlist_project'].options, prefer_pandas=False)
print(to_print)


Contents of `inlist`:
namelist | name                          | default     | current          | description                                                 
---------+-------------------------------+-------------+------------------+-------------------------------------------------------------
star_job | read_extra_star_job_inlist(1) | .false.     | .true.           | You can split your ``star_job`` inlist into pieces using the following controls. BTW: it works re...
star_job | extra_star_job_inlist_name(1) | 'undefined' | 'inlist_project' |                                                             
eos      | read_extra_eos_inlist(1)      | .false.     | .true.           | One can split an eos inlist into pieces using the following parameters. It works recursively, so ...
eos      | extra_eos_inlist_name(1)      | 'undefined' | 'inlist_project' |                                                             
kap      | read_extra_kap_inlist(1)      | .false.     | .true.           | 

In [7]:
initial_masses = np.arange(10, 18, 4)
overshoot_fs = np.arange(0.004, 0.016, 0.004)

name_template = 'm%s_fov%s'
model_dict = {}

for mi in initial_masses:
    for fov in overshoot_fs:
        name = name_template % (int(mi), str(fov).replace('.', 'p'))
        print(name)
        this_model = base_model.copy(model_copy_name=name, deep_copy=True)
        this_model.inlists['inlist_project'].set_options(defaults=def_opts, controls={'initial_mass': mi,
                                                                                      'overshoot_f(1)': fov})
        model_dict[name] = this_model

m10_fov0p004
m10_fov0p008
m10_fov0p012
m14_fov0p004
m14_fov0p008
m14_fov0p012


Although it would be more efficient to immediately wrap the resulting `Model` objects inside `GridRunItem` objects and create the necessary model directories, it is wiser to store the `Model` objects somewhere else (in this case in a dictionary) first. This allows us to inspect them to see if the requested changes to the inlists worked out as intended, before writign anything to our system storage. 

In [ ]:
print('Contents of `inlist_project`:')
to_print = morpy.preview(model_dict['m14_fov0p012'].inlists['inlist_project'].options, prefer_pandas=False)
print(to_print)

Contents of `inlist_project`:
namelist | name                              | default | current       | description                                                 
---------+-----------------------------------+---------+---------------+-------------------------------------------------------------
star_job | create_pre_main_sequence_model    | .false. | .true.        | If true, the code will create a starting model with uniform composition, a core temperature below...
star_job | save_model_when_terminate         | .false. | .false.       | save final model when a run terminates only happens if satisfy required termination code if strin...
star_job | pgstar_flag                       | .false. | .false.       | if true, activates pgplot output                            
kap      | use_Type2_opacities               | .true.  | .true.        | Select whether to use Type2 opacity tables (see :ref:`kap/defaults:kap_CO_prefix`). Even when tru...
kap      | Zbase                             |

In [ ]:
# Create a project directory
project_dir = morpy.create_projectdir(Path('/Users/janhenneco/GitHub/morpy/docs/examples/example_output/run_a_grid/project_dir'), mode='star', config=mconf)

# Create model directories for all models, add the models to GridRunItem wrappers and add those to a list
grid_list = []
for key in model_dict.keys():
    model = model_dict[key]
    grid_list.append(morpy.GridRunItem(model=model))
    morpy.create_modeldir(model=model, projectdir_path=project_dir, overwrite=True)

print(f'{len(grid_list)} models prepared.')
for item in grid_list:
    print(f'  {item.model.name}  --->  {item.model.model_path}')

# Compile the executables
compile_result = morpy.compile_project(projectdir_path=project_dir, mode='star')
print(compile_result.pretty(2000))

6 models prepared.
  m10_fov0p004  --->  /Users/janhenneco/GitHub/mesa-runner/docs/examples/example_output/run_a_grid/project_dir/m10_fov0p004
  m10_fov0p008  --->  /Users/janhenneco/GitHub/mesa-runner/docs/examples/example_output/run_a_grid/project_dir/m10_fov0p008
  m10_fov0p012  --->  /Users/janhenneco/GitHub/mesa-runner/docs/examples/example_output/run_a_grid/project_dir/m10_fov0p012
  m14_fov0p004  --->  /Users/janhenneco/GitHub/mesa-runner/docs/examples/example_output/run_a_grid/project_dir/m14_fov0p004
  m14_fov0p008  --->  /Users/janhenneco/GitHub/mesa-runner/docs/examples/example_output/run_a_grid/project_dir/m14_fov0p008
  m14_fov0p012  --->  /Users/janhenneco/GitHub/mesa-runner/docs/examples/example_output/run_a_grid/project_dir/m14_fov0p012
success: True
return_code: 0
duration_seconds: 1.7723829579990706
mode: star
executable_path: /Users/janhenneco/GitHub/mesa-runner/docs/examples/example_output/run_a_grid/project_dir/star
model_scripts_updated: ['/Users/janhenneco/GitHub

## 5. Enabling tracking functionalities

Before we launch the grid, we will set up the necessary machinery to keep track of the status of our models. This machinery incldues three components.

The `ModelRepository` is the object that creates and points to a SQLite database file stored at `DB_PATH`. This file holds all the necessary information of the status of a model and its history. Even when the current Python session is terminated, the SQLite database file will remain where we first created it, and can be loaded again. In fact, if we store model information and later decide to rerun the grid using the same database file, we need to remove it first, since calling `ModelRepository(db_path=DB_PATH)` again will not overwrite existing files at `DB_path`.

The `ModelObserver` is an instance that is constantly "listening" for any changes to the status of models (e.g. from running ---> finished) and record it in the SQLite database file.

The `LiveModelTracker` is an instance tracks live process handles for model objects currently in the RAM and loses its function if the Python session is closed. 

In [ ]:
# Path to the SQLite database file that stores information about models (important for tracking).
DB_PATH = project_dir / 'runs.db'

# Remove the existing file at DB_PATH to start clean
if DB_PATH.exists():
    DB_PATH.unlink()

print(DB_PATH)

repo = morpy.ModelRepository(db_path=DB_PATH)
observer = morpy.ModelObserver(repo=repo)
live_tracker = morpy.LiveModelTracker()

# Adding the models to the repo befoe running allows us to have a more general overview
models = [item.model for item in grid_list]
repo.add_models_to_repo(models=models)


/Users/janhenneco/GitHub/mesa-runner/docs/examples/example_output/run_a_grid/project_dir/runs.db


## 6. Running and launching the grid

Now we'll see how to run (blocking) and launch (non-blocking) the grid we just set up. For this purpose, the `run_grid()` and `launch_grid()` function were created. Both functions take `backend` as an argument, which allows us to choose which parallelisation backend to use. The `local` backend uses the built-in `subprocesses` package to run multiple models in parallel and is great for single-node computations. The `mpi` backend uses `MPI` via `mpi4pi` to run or launch the grid. Using this backend requires the function to be called in a specific way, which we will leanr about in the `run_a_grid_with_mpi` example.

As before, if we run the grid, we will have to wait until the grid has completely finished before we can continue working in our current Python session. If the Python session is closed, grid computations started by the `run_grid()` and `launch_grid()` functions will stop working, although some of the currently running models will likely continiue runnign until they are finished. Nevertheless, this behavior is undesireable. The main benefit of launching the grid vs. running it, is that we can launch the tracking functionalities set up above within the same Python session.  

In [ ]:
grid_handle = morpy.launch_grid(grid_list,
                            backend='local',
                            max_workers=3, # Number of models to run simultaneously
                            sync_before_run=True, # Sync any changes made to inlist files with Model objects in memory
                            default_options=def_opts,
                            config=mconf,
                            capture_output=True, # Store terminal output in individual models' RunResults
                            observer=observer,
                            live_tracker=live_tracker
                            ) 

vars(grid_handle)

{'grid_id': 'a01b7dd822094afdae226992e1b90fd6',
 'backend': 'local',
 'submitted_at': datetime.datetime(2026, 6, 17, 8, 57, 9, 180214, tzinfo=datetime.timezone.utc),
 'future': <Future at 0x1087e6d90 state=running>,
 '_executor': <concurrent.futures.thread.ThreadPoolExecutor at 0x1087e5890>}

In [ ]:
grid_result = morpy.wait_for_grid(grid_handle)

In [ ]:
vars(grid_result)

In [ ]:
morpy.print_grid_status(repo=repo, live_tracker=live_tracker)

[2026-06-17 08:57:43 UTC +00:00]

MODEL         STATE      PID  RC
------------  -------  -----  --
m10_fov0p004  running  81881    
m10_fov0p008  running  81882    
m10_fov0p012  running  81883    
m14_fov0p004  draft             
m14_fov0p008  draft             
m14_fov0p012  draft             


In [ ]:
morpy.stop_grid(handle=grid_handle, live_tracker=live_tracker, force=True)

{'grid_id': 'a01b7dd822094afdae226992e1b90fd6',
 'future_cancelled': False,
 'future_done': True,
 'terminated_model_ids': []}